### Dataset used for training

https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset


In [1]:
print("ok")

ok


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import os
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    BatchNormalization,
    Flatten,
    MaxPool2D
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

### Data Preprocessing

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    'dataset/training_set',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    'dataset/test_set',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

train_data.class_indices

Found 20000 images belonging to 2 classes.
Found 400 images belonging to 2 classes.


{'cats': 0, 'dogs': 1}

### Function to predict the class of an image

In [9]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

def calculate_accuracy(
    class0_path,
    class1_path,
    model_path,
    target_size=(128, 128)
):

    model = load_model(model_path)
    
    model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

    correct = 0
    total = 0

    for file in os.listdir(class0_path):

        img_path = os.path.join(class0_path, file)

        try:
            img = image.load_img(
                img_path,
                target_size=target_size
            )

            img_array = image.img_to_array(img)

            img_array = np.expand_dims(img_array, axis=0)

            img_array = img_array / 255.0

            prediction = model.predict(img_array, verbose=0)

            predicted_class = 1 if prediction[0][0] > 0.5 else 0

            if predicted_class == 0:
                correct += 1

            total += 1

        except Exception as e:
            print(f"Error processing {img_path}: {e}")

    for file in os.listdir(class1_path):

        img_path = os.path.join(class1_path, file)

        try:
            img = image.load_img(
                img_path,
                target_size=target_size
            )

            img_array = image.img_to_array(img)

            img_array = np.expand_dims(img_array, axis=0)

            img_array = img_array / 255.0

            prediction = model.predict(img_array, verbose=0)

            predicted_class = 1 if prediction[0][0] > 0.5 else 0

            if predicted_class == 1:
                correct += 1

            total += 1

        except Exception as e:
            print(f"Error processing {img_path}: {e}")


    accuracy = (correct / total) * 100

    print(f"\nTotal Images: {total}")
    print(f"Correct Predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")

    return accuracy

### Base Model

In [6]:
model1=Sequential()
from keras.layers import Conv2D
model1.add(Conv2D(input_shape=[128,128,3], filters=32, kernel_size=3, activation='relu'))
model1.add(MaxPool2D(pool_size=2,strides=2))
model1.add(Flatten())
model1.add(Dense(units=64,activation='relu'))
model1.add(Dense(units=1,activation='sigmoid'))
model1.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model1.fit(x=train_data, validation_data=val_data, epochs=20)
model1.save('model1.h5')

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 141s 223ms/step - accuracy: 0.6562 - loss: 0.6701 - val_accuracy: 0.6975 - val_loss: 0.5500
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 119s 190ms/step - accuracy: 0.7085 - loss: 0.5564 - val_accuracy: 0.7125 - val_loss: 0.5502
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 181s 290ms/step - accuracy: 0.7244 - loss: 0.5403 - val_accuracy: 0.7250 - val_loss: 0.5434
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 122s 195ms/step - accuracy: 0.7379 - loss: 0.5247 - val_accuracy: 0.7500 - val_loss: 0.5017
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 125s 199ms/step - accuracy: 0.7524 - loss: 0.5063 - val_accuracy: 0.7625 - val_loss: 0.4893
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 239s 383ms/step - accuracy: 0.7528 - loss: 0.5023 - val_accuracy: 0.7725 - val_loss: 0.4853
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 307s 492ms/step - accuracy: 0.7603 - loss: 0.4908 - val_accuracy: 0.7950 - val_loss: 0.4766
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 294s 471ms/step - accuracy: 0.7674 -

In [8]:
accuracy = calculate_accuracy(
    class0_path='dataset/predication/cats',
    class1_path='dataset/predication/dogs',
    model_path='model1.h5'
)


Total Images: 8048
Correct Predictions: 6572
Accuracy: 81.66%


### Model 2
Added Dropout Layers and Early Stopping to further improve model accuracy. 

In [12]:
model2 = Sequential()

model2.add(
    Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
        input_shape=(128,128,3)
    )
)

model2.add(BatchNormalization())

model2.add(
    MaxPool2D(
        pool_size=2,
        strides=2
    )
)

model2.add(Dropout(0.25))

model2.add(
    Conv2D(
        filters=64,
        kernel_size=3,
        activation='relu'
    )
)

model2.add(BatchNormalization())

model2.add(
    MaxPool2D(
        pool_size=2,
        strides=2
    )
)

model2.add(Dropout(0.25))

model2.add(Flatten())

model2.add(Dense(units=128, activation='relu'))

model2.add(Dropout(0.5))

model2.add(Dense(units=1, activation='sigmoid'))

model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
model2.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    callbacks=[early_stop]
)
model2.save('model2.h5')

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 237s 376ms/step - accuracy: 0.5663 - loss: 0.9827 - val_accuracy: 0.5900 - val_loss: 2.4291
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 250s 399ms/step - accuracy: 0.5867 - loss: 0.6663 - val_accuracy: 0.5575 - val_loss: 0.6656
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 244s 390ms/step - accuracy: 0.6098 - loss: 0.6434 - val_accuracy: 0.5775 - val_loss: 0.6649
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 247s 396ms/step - accuracy: 0.6295 - loss: 0.6285 - val_accuracy: 0.6875 - val_loss: 0.6099
Epoch 5/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 238s 381ms/step - accuracy: 0.6589 - loss: 0.6025 - val_accuracy: 0.6775 - val_loss: 0.6149
Epoch 6/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 252s 404ms/step - accuracy: 0.6657 - loss: 0.5997 - val_accuracy: 0.6950 - val_loss: 1.4862
Epoch 7/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 259s 414ms/step - accuracy: 0.6874 - loss: 0.5850 - val_accuracy: 0.7400 - val_loss: 0.5529
Epoch 8/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 260s 416ms/step - accuracy: 0.7050 -

In [13]:
accuracy = calculate_accuracy(
    class0_path='dataset/predication/cats',
    class1_path='dataset/predication/dogs',
    model_path='model2.h5'
)


Total Images: 8048
Correct Predictions: 6560
Accuracy: 81.51%


### Model3 Transfer Learning

As Model 2 accuracy is still not good. So, we will use transfer learning to improve the performance of the model. We will use a pre-trained model as the base model and add a few layers on top of it to fine-tune the model for our specific task.

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# =========================
# LOAD PRETRAINED MODEL
# =========================

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze pretrained layers
base_model.trainable = False

# =========================
# BUILD MODEL
# =========================

model3 = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

# =========================
# COMPILE MODEL
# =========================

model3.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# EARLY STOPPING
# =========================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# =========================
# TRAIN MODEL
# =========================

history = model3.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop]
)

# =========================
# FINE TUNING
# =========================

base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

model3.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model3.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

# =========================
# SAVE MODEL
# =========================

model3.save("model3.h5")



9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 135s 210ms/step - accuracy: 0.9381 - loss: 0.1524 - val_accuracy: 0.9500 - val_loss: 0.1181
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 129s 207ms/step - accuracy: 0.9533 - loss: 0.1185 - val_accuracy: 0.9650 - val_loss: 0.1113
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 131s 210ms/step - accuracy: 0.9557 - loss: 0.1116 - val_accuracy: 0.9700 - val_loss: 0.0889
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 134s 215ms/step - accuracy: 0.9571 - loss: 0.1070 - val_accuracy: 0.9700 - val_loss: 0.0876
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 134s 214ms/step - accuracy: 0.9574 - loss: 0.1074 - val_accuracy: 0.9700 - val_loss: 0.0986
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 138s 221ms/step - accuracy: 0.9588 - loss: 0.1006 - val_accuracy: 0.9525 - val_loss: 0.1262
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 131s 210ms/step - accuracy: 0.9596 - loss: 0.0978 - val_accuracy: 0.9700 - val_loss: 0.1012
Epoch 8/20
625/625 ━━━━━━━━

In [15]:
accuracy = calculate_accuracy(
    class0_path='dataset/predication/cats',
    class1_path='dataset/predication/dogs',
    model_path='model3.h5'
)


Total Images: 8048
Correct Predictions: 7947
Accuracy: 98.75%
